In [23]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Round structure
[start with preset capitals] Prodcution -> Market Deals-> Export Deals -> Results of a round [workplace changes, firms analytics] -> End of a round
## New cycle
Prodcution -> Market Deals -> Results of a round -> Export Deals -> End of a round

## Initialization process:
Resolver->Markets->Agents

In [25]:
# Make 0 for unemployed


#can`t find a good reason to use this...

# class Workplace:
#     def __init__(self, inner_sector, cid, pmnt):
#         assert (inner_sector == True or inner_sector == False), "'inner_sector' must be boolean"
#         self.inner_sector = inner_sector
#         self.company_id   = cid
#         self.payment      = pmnt

#     def payment(self):
#         return self.payment
        

1. Позволить домохозяйствам работать вне НГ сектора
2. Обновление self.madness у домохозяйств при неудовлетворенной функции полезности
3. Обязательное увеличение зп, если штат не набирается 3 шага
4. Добавить self.location для домохозяйств, ограничивающее мобильность смены места работы
5. У меня сайлайнсер в долги влез
6. Нормальные проценты того сколько и кто тратит
7. Платить зарплаты

In [26]:
class Firm:
    ...
class Household:
    ...


class GlobalResolver:
    ''' Stores and updates dictionaries of simulation; \n Notifies firms and households; \n Completes transactions; \n Shows information about active agents; \n '''

    def __init__(self, model, logfile_name='test'):

        self.model      = model

        
        self.data_log = pd.DataFrame(columns=['Facility', 'Message Type', 'Message_txt' ])
        self.companies  = dict()
        self.households = dict()
        
        self._comp_id   = 1
        self._house_id  = 1

                # буффер лога
        self._buffer_log = []
        
        self._facility_ids = {
            1: 'Household',
            2: 'Firm',
            3: 'Goods Market',
            4: 'Labour Market',
            5: 'Outer region'
        }
        
        self._event_ids = {
            1: 'Job offer listed',
            2: 'Extraction completed',
            3: 'Goods listed',
            4: 'Sales completed',
            5: 'Worplace change: success',
            6: 'All staff recruited',
            7: 'Consumed',
            8: 'Export completed',
            9: 'Out of stock'
        }

    # заполнение буфера
    def buffer_log(self, facility_id, event_id, data_cort):
        ''' 
        Id's for Facility logging:
        -------------------------
        'Household',        | 1 |
        'Firm',             | 2 |
        'Goods Market',     | 3 |
        'Labour Market',    | 4 |
        'Outer region'      | 5 |
        -------------------------

        Id's for Event logging:
        -------------------------------
        'Job offer listed',          | 1 |
        'Extraction completed',      | 2 |
        'Goods listed',              | 3 |
        'Sales completed',           | 4 |
        'Worplace change: success',  | 5 |
        'All staff recruited'        | 6 |
        'Consumed',                  | 7 |
        'Export completed',          | 8 |
        'Out of stock'               | 9 |
        
        '''
        
        self._buffer_log.append((
            facility_id,
            event_id,
            data_cort
        ))

    def decode_buffer(self):
        decoded = []
    
        for facility_id, event_id, data_cort in self._buffer_log:
            facility = self._facility_ids.get(facility_id, f'Unknown({facility_id})')
            event = self._event_ids.get(event_id, f'Unknown({event_id})')
    
            # форматирование сообщений
            if event_id == 2:  # Extraction completed
                cid, q, cost = data_cort
                message = f'Id: {cid} Q: {q:.2f} Cost: {cost:.2f}'
    
            elif event_id == 4:  # Sales completed
                cid, profit = data_cort
                message = f'Id: {cid} Profit: {profit:.2f}'
    
            elif event_id == 1:  # Job offer listed
                cid, vacancies, payment = data_cort
                message = f'Id: {cid} vacancies={vacancies} payment={payment}'
    
            elif event_id == 5:  # Workplace change
                hid, cid = data_cort
                message = f'Id: {hid} changed workplace to {cid}'
    
            elif event_id == 6:  # All staff recruited
                cid = data_cort
                message = f'Id: {cid}'
    
            elif event_id == 3:  # Goods listed
                cid, q, price = data_cort
                message = f'Id: {cid} listed {q} Price: {price}'
    
            elif event_id == 7:  # Consumed
                hid, q, cid, total = data_cort
                message = f'Id: {hid} bought {q} from {cid} Total: {total}'
    
            elif event_id == 9:  # Out of stock
                cid  = data_cort
                message = f'Id: {cid}'
    
            elif event_id == 8:  # Export completed (если появится формат)
                message = f'data={data_cort}'
    
            else:
                message = f'data={data_cort}'
    
            decoded.append((facility, event, message))
    
        return decoded
    
    def flush_to_csv(self, filepath):
        if not self._buffer_log:
            return
    
        decoded = self.decode_buffer()
    
        df = pd.DataFrame(decoded, columns=[
            'Facility',
            'Message Type',
            'Message_txt'
        ])
    
        df.to_csv(filepath, mode='a', header=not os.path.exists(filepath), index=False)
    
        self._buffer_log.clear()

        

    def add_firm(self, Firm: Firm) -> int:
        
        self.companies[self._comp_id] = Firm
        self._comp_id += 1
        
        return int(self._comp_id - 1) 
    
    def add_household(self, Household: Household) -> int:
        self.households[self._house_id] = Household
        self._house_id += 1
        
        return int(self._house_id - 1) 
        

    
    def comlete_sale(self, company_id, household_id, price, quantity):
        '''Transfers money from a household 
        to firm'''
        
        self.companies[company_id].capital    += price*quantity
        self.households[household_id].savings -= price*quantity
        
        self.companies[company_id].log_solded(quantity)
        

    
    def complete_hiring(self, company_id, household_id, payment):
        '''Gives information about new job to the household'''
        
        self.households[household_id].workplace_id = company_id
        self.households[household_id].payment      = payment

    
    def calculate_manhours(self):
        '''Calulates manhours for all firms'''
        for h_id in self.households:
            company_id = int(self.households[h_id].workplace_id)
            self.companies[company_id].avaliable_man_hours += 160

    def step_before_Households(self):
        for indx in self.companies:
            self.companies[indx].step_before_Households()

    def step_after_Households(self):
        for indx in self.companies:
            self.companies[indx].step_after_Households()

    def step_consume_Households(self):
        for indx in self.households:
            self.households[indx].consume()

    def step_workchange_Households(self):
        for indx in self.households:
            self.households[indx].workchange()
        
    

    
    def households_info(self):
        '''Returns DataFrame with basic info for each household'''
        dfs = []
        for h_id in self.households:
            dfs.append(self.households[h_id].info())

        return pd.concat(dfs)

    
    def companies_info(self) -> pd.DataFrame:
        '''Returns DataFrame with basic info for each firm'''
            
        dfs = []
        for c_id in self.companies:
            workerks = 0
            df = self.companies[c_id].info()
            df['Workers'] = 0 
            
            for h_id in self.households:
                if self.households[h_id].workplace_id == c_id:
                    df['Workers'] += 1

            dfs.append(df)
                
        return pd.concat(dfs)

    def log(self, message_cort):
        '''Ypu need to pass a message cortege (Facility(Household/Firm/Goods Market/Labour Market), Message type( 'All staff recruited', 'Worplace change: success',  'Wokrplace change: failure', 'Job offer listed', 'Out of stock', 'Consumed', 'Goods listed'), Message_txt). \n This data will be saved in log'''
        ''' 
        Id's for Facility logging:
        -------------------------
        'Household',        | 1 |
        'Firm',             | 2 |
        'Goods Market',     | 3 |
        'Labour Market',    | 4 |
        'Outer region'      | 5 |
        -------------------------

        Id's for Event logging:
        -------------------------------
        'Job offer listed',          | 1 |
        'Extraction completed',      | 2 |
        'Goods listed',              | 3 |
        'Sales completed',           | 4 |
        'Worplace change: success',  | 5 |
        'All staff recruited'        | 6 |
        'Consumed',                  | 7 |
        'Export completed',          | 8 |
       
        '''
        #/ assert message_cort[0] in ['Household', 'Firm', 'Goods Market', 'Labour Market', 'Outer region']
       
        # assert message_cort[1] in [
        #     'All staff recruited', 'Worplace change: success',  'Wokrplace change: failure'
        #     'Job offer listed', 'Out of stock', 'Consumed', 'Goods listed', 'Export completed',
        #     'Production completed'
        # ]
        
        self.data_log.loc[len(self.data_log)] = message_cort

    def show_log(self) -> pd.DataFrame:
        # '''Save log pd.DataFrame as a .pkl file with seratain name. If name isn`t passed: it will automaticly add a number to a setted in initialization name '''

        return self.data_log




In [27]:
d = {}
d[1] = 2
d[2] = 3
d

{1: 2, 2: 3}

In [28]:
del d[1]
d

{2: 3}

In [29]:
import numpy as np

START_MEAN_WAGE = 102000

class LabourMarket:
    def __init__(self, resolver):
        self.resolver = resolver

        # динамические массивы
        self.company_ids = np.empty(0, dtype=np.int64)
        self.payments    = np.empty(0, dtype=np.float64)
        self.vacancies   = np.empty(0, dtype=np.int64)

        # быстрый доступ по id
        self.id_to_idx = {}

    # --- добавление вакансии ---
    def push(self, company_id: int, payment, vacancies):

        if company_id in self.id_to_idx:
            # обновляем существующую запись
            idx = self.id_to_idx[company_id]
            self.payments[idx] = payment
            self.vacancies[idx] += vacancies
        else:
            # добавляем новую
            idx = len(self.company_ids)

            self.company_ids = np.append(self.company_ids, company_id)
            self.payments    = np.append(self.payments, payment)
            self.vacancies   = np.append(self.vacancies, vacancies)

            self.id_to_idx[company_id] = idx

        # лог
        # self.resolver.log(
        #     message_cort=(
        #         'Labour Market',
        #         'Job offer listed',
        #         f'Id: {company_id} vacancies={vacancies} payment={payment}'
        #     )
        # )

        self.resolver.buffer_log(
            facility_id=4, 
            event_id=1, 
            data_cort=(company_id, vacancies, payment)
        )

    # --- найм ---
    def get_job(self, company_id, household_id):

        idx = self.id_to_idx.get(company_id)
        if idx is None:
            return  # или ошибка

        # уменьшаем вакансии
        self.vacancies[idx] -= 1
        payment = self.payments[idx]

        # завершение найма
        self.resolver.complete_hiring(company_id, household_id, payment)

        # self.resolver.log(
        #     message_cort=(
        #         'Household',
        #         'Worplace change: success',
        #         f'Id: {household_id} changed workplace to {company_id}'
        #     )
        # )

        # GlobalResolver.buffer_log()
        self.resolver.buffer_log(
            facility_id=1, 
            event_id=5, 
            data_cort=(household_id, company_id)
        )

        # если вакансий больше нет → удаляем
        if self.vacancies[idx] <= 0:
            self._remove(idx)

            # self.resolver.log(
            #     message_cort=(
            #         'Firm',
            #         'All staff recruited',
            #         f'Id: {company_id}'
            #     )
            # )

            
            # GlobalResolver.buffer_log()
            self.resolver.buffer_log(
                facility_id=2, 
                event_id=6, 
                data_cort=(company_id)
            )

    # --- удаление строки ---
    def _remove(self, idx):
        # id_to_idx = {Company id : array's index}
        # deleting a line with index = id
        deleting_id = self.company_ids[idx]
        last = len(self.company_ids) - 1
        last_id = self.company_ids[last]

        # swap с последним элементом
        self.company_ids[idx] = self.company_ids[last]
        self.payments[idx]    = self.payments[last]
        self.vacancies[idx]   = self.vacancies[last]

        # обновляем индекс
        self.id_to_idx[last_id] = idx        

        # обрезаем массивы
        self.company_ids = self.company_ids[:-1]
        self.payments    = self.payments[:-1]
        self.vacancies   = self.vacancies[:-1]

        # удаляем старый id
        del self.id_to_idx[deleting_id]

    # --- получить офферы ---
    def get_offers(self, amount: int):

        n = len(self.company_ids)

        if n <= amount:
            return self.company_ids, self.payments, self.vacancies

        idx = np.random.choice(n, size=amount, replace=False)

        return (
            self.company_ids[idx],
            self.payments[idx],
            self.vacancies[idx]
        )

    # --- средняя зарплата ---
    def get_mean_wage(self):

        if len(self.payments) == 0:
            return START_MEAN_WAGE

        return self.payments.mean()

    # --- полный доступ ---
    def actual_data(self):
        return pd.DataFrame({ 'Company Id': self.company_ids, 'Payment' : self.payments, 'Vacancies' : self.vacancies})

In [30]:
import numpy as np

class GoodsMarket:

    def __init__(self, resolver, logging, no_compete_sale=False):
        self.resolver = resolver
        self.logging  = logging
        self.no_compete_sale = no_compete_sale

        self.company_ids = np.empty(0, dtype=np.int64)
        self.prices      = np.empty(0, dtype=np.float64)
        self.quantities  = np.empty(0, dtype=np.float64)
        self.prestigue   = np.empty(0, dtype=np.float64)

        self.id_to_idx = {}
        

    # --- добавление товара ---
    def push(self, company_id: int, price: float, quantity: float, prestigue: float):

        if company_id in self.id_to_idx:
            idx = self.id_to_idx[company_id]
            self.prices[idx] = price
            self.quantities[idx] += quantity
            self.prestigue[idx] = prestigue
        else:
            idx = len(self.company_ids)

            self.company_ids = np.append(self.company_ids, company_id)
            self.prices      = np.append(self.prices, price)
            self.quantities  = np.append(self.quantities, quantity)
            self.prestigue   = np.append(self.prestigue, prestigue)

            self.id_to_idx[company_id] = idx

        # self.resolver.log(
        #     message_cort=(
        #         'Goods Market',
        #         'Goods listed',
        #         f'Id: {company_id} listed {quantity} Price: {price}'
        #     )
        # )
        
        self.resolver.buffer_log(
            facility_id=3,
            event_id=3,
            data_cort=(company_id, quantity, price)
        )

    # --- покупка ---
    def buy(self, company_id: int, household_id: int, quantity: float):

        idx = self.id_to_idx.get(company_id)
        if idx is None:
            return

        price = self.prices[idx]

        # ограничение по наличию
        actual_q = min(quantity, self.quantities[idx])

        # списание товара
        self.quantities[idx] -= actual_q

        total_cost = price * actual_q

        self.resolver.buffer_log(
            facility_id=3,
            event_id=7,
            data_cort=(household_id, actual_q, cid, price)
        )
        # self.resolver.log(
        #     message_cort=(
        #         'Goods Market',
        #         'Consumed',
        #         f'Id: {household_id} bought {actual_q} from {company_id} Total: {total_cost}'
        #     )
        
        # )

        # завершение сделки
        self.resolver.comlete_sale(company_id, household_id, price, actual_q)

        # если закончился товар → удаляем
        if self.quantities[idx] <= 0:
            self._remove(idx)

            self.resolver.buffer_log(
                facility_id=2,
                event_id=9,
                data_cort=(company_id)
            )
            # self.resolver.log(
            #     message_cort=(
            #         'Firm',
            #         'Out of stock',
            #         f'Id: {company_id}'
            #     )
            # )

    def bulk_buy(self, company_ids, household_id, quantities):
        """
        company_ids: np.array
        quantities:  np.array
        """
    
        # переводим в индексы
        idx = np.fromiter((self.id_to_idx[cid] for cid in company_ids), dtype=int)
    
        prices = self.prices[idx]
    
        # ограничение по наличию
        actual_q = np.minimum(quantities, self.quantities[idx])
    
        # списание
        self.quantities[idx] -= actual_q
    
        # стоимость
        costs = actual_q * prices
    
        # --- логирование ---
        if (self.logging):
            for i, cid in enumerate(company_ids):
                q = actual_q[i]
                if q > 0:
                    self.resolver.buffer_log(facility_id=3, event_id=7, data_cort=(household_id, q, cid, prices[i]))
                    # self.resolver.log(
                    #     message_cort=(
                    #         'Goods Market',
                    #         'Consumed',
                    #         f'Id: {household_id} bought {q} from {cid} Total: {q * prices[i]}'
                    #     )
                    # )
    
        if (self.no_compete_sale == False):
            for i, cid in enumerate(company_ids):
                q = actual_q[i]
                if q > 0:
                    self.resolver.comlete_sale(
                        company_id=cid,
                        household_id=household_id,
                        price=prices[i],
                        quantity=q
                    )
    
        return actual_q, costs
        
    # --- удаление (O(1)) ---
    def _remove(self, idx):

        # id_to_idx = {Company id : array's index}
        # deleting a line with index = id
        deleting_id = self.company_ids[idx]
        last = len(self.company_ids) - 1
        last_id = self.company_ids[last]

        # swap
        self.company_ids[idx] = self.company_ids[last]
        self.prices[idx]      = self.prices[last]
        self.quantities[idx]  = self.quantities[last]
        self.prestigue[idx]   = self.prestigue[last]

        self.id_to_idx[last_id] = idx

        # обрезка
        self.company_ids = self.company_ids[:-1]
        self.prices      = self.prices[:-1]
        self.quantities  = self.quantities[:-1]
        self.prestigue   = self.prestigue[:-1]

        del self.id_to_idx[deleting_id]

    # --- получение предложений ---
    def get_offers(self, amount: int, mode='random', companies_set=None):

        n = len(self.company_ids)

        if n <= amount:
            return self.company_ids, self.prices, self.quantities, self.prestigue

        if mode == 'random':
            idx = np.random.choice(n, size=amount, replace=False)

        elif mode == 'include':
            assert companies_set is not None and len(companies_set) > 0

            include_mask = np.isin(self.company_ids, list(companies_set))

            include_idx = np.where(include_mask)[0]
            exclude_idx = np.where(~include_mask)[0]

            need = amount - len(include_idx)

            if need > 0:
                extra = np.random.choice(exclude_idx, size=need, replace=False)
                idx = np.concatenate([include_idx, extra])
            else:
                idx = include_idx[:amount]
        else:
            raise ValueError("mode must be 'random' or 'include'")

        return (
            self.company_ids[idx],
            self.prices[idx],
            self.quantities[idx],
            self.prestigue[idx]
        )

    # --- все данные ---
    def actual_data(self):
        return pd.DataFrame({'Company Id':self.company_ids, 'Price' : self.prices, 'Quantities' : self.quantities, 'Presigue':self.prestigue})

In [31]:
from scipy.special import softmax
#it's used to destribute spendings of a househould between companies
# as a temperature it uses madness^{-1}  


# Endogeneous variables
GOV_SUPPORT   = ...    
MULTIPLY_COST = ... 
BASE_JOB_OFFERS_AMOUNT = 3

#Might be personal for each agent, but for now they will be fixed
ALPHA = 123
BETA  = 31
DELTA = 200


#what about sizes of variables
#NOTE: There is no realisation of elastisity for company change (in case if demand is not satisfied)
# def U_j(prestigue, preference, price):
#     return ALPHA*np.log(prestigue) + DELTA*preference - BETA*np.log(price)

#needs testing
def U_numpy(deals: dict, company_ids, prices, prestigue, madness: float, savings):

    # --- 1. consumption ---
    k = np.clip((madness + 3) / 10, 0.3, 0.9)

    consumption = k * savings

    # --- 2. preferences ---
    n = len(prices)

    if len(deals) > 0:
        prefs = np.array([deals.get(cid, 0.0) for cid in company_ids])
        total = prefs.sum()
        if total > 0:
            prefs = prefs / total
        else:
            prefs = np.ones(n) / n
    else:
        prefs = np.ones(n) / n

    # --- 3. utility ---
    X = ALPHA * np.log(prestigue) + DELTA * prefs - BETA * np.log(prices)

    # --- 4. softmax ---
    madness = max(madness, 0.1)

    logits = X / madness
    logits -= logits.max()   # стабилизация

    exp_logits = np.exp(logits)
    destri = exp_logits / exp_logits.sum()

    # --- 5. деньги → количество ---
    spendings = destri * consumption
    quantities = spendings / prices

    return quantities
        

    #adds 'temperature'
    logits_with_pref = np.array(X) / madness 
    #calculated prefered destribution for money between companies
    destri = softmax(logits_with_pref)
    
    #U_{i} = P_{i} * C 
    spendings = destri * consumption
    quantities = spendings / avaliable_goods["Price"].values
    
    avaliable_goods["Demand"] = quantities

    return avaliable_goods

class Household:
    def __init__(self, start_savings, company_id, payment, model, brain_damage=6.0):

        #Sumulation classes
        self.model = model
        
        #base parametres
        self.savings       = start_savings
        self.workplace_id  = int(company_id)
        self.payment       = payment
        self._brain_damage = brain_damage
        self.madness       = np.random.normal(loc=0.0, scale=brain_damage)
        self.id            = self.model.resolver.add_household(self)
        

        #knowledge
        self.deals         = dict()
        

    def info(self):
        data = pd.DataFrame(
            [{
                'Id'            : self.id,
                "Savings"       : self.savings,
                "Company ID"    : self.workplace_id,
                "Payment"       : self.payment,
                "Activeness"    : self.madness
                
            }])
        return data

    def set_id(self, indx):
        self.id = indx
    
    def search_goods(self):
        '''Performs search based on self._madness (as a probability to find new offers) and self.deals \n Note: Every agent on each iteration remembers n=BASE_JOB_OFFERS_AMOUNT most familliar to it companies (the ones it traded the most).\n With help of it`s own self.madness it can seen more than n=BASE_JOB_OFFERS_AMOUNT avaliable deals'''
        
        seen_firms = self.deals.keys()

        amount = int(max(self.model.BASE_JOB_OFFERS_AMOUNT, abs(self.model.BASE_JOB_OFFERS_AMOUNT+self.madness)))
        
        if len(seen_firms) == 0:
            return self.model.GoodsMarket.get_offers(amount, mode='random')
        else:
            return self.model.GoodsMarket.get_offers(amount, mode='include', companies_set=seen_firms)

            
    def consume(self):
    
        company_ids, prices, quantities, prestigue = self.search_goods()
    
        if len(company_ids) == 0:
            return
    
        demand = U_numpy(
            deals=self.deals,
            company_ids=company_ids,
            prices=prices,
            prestigue=prestigue,
            madness=self.madness,
            savings=self.savings
        )
    
        actual_q = np.minimum(demand, quantities)
    
        costs = actual_q * prices
        total_cost = costs.sum()
    
        # if total_cost > self.savings:
        #     scale = self.savings / total_cost
        #     actual_q *= scale
        #     costs *= scale
    
        actual_q, costs = self.model.GoodsMarket.bulk_buy(
            company_ids,
            self.id,
            actual_q
        )
    
        # self.savings -= costs.sum()
    
        # обновляем память
        for i, cid in enumerate(company_ids):
            q = actual_q[i]
            if q > 0:
                self.deals[cid] = self.deals.get(cid, 0) + q
                
    def search_work_offers(self):
        '''performs search based on self._madness (as a probability to find new offers)'''

        amount = int(max(self.model.BASE_JOB_OFFERS_AMOUNT, abs(self.model.BASE_JOB_OFFERS_AMOUNT+self.madness)))

        return self.model.LabourMarket.get_offers(amount=amount)


    #BAD method to make firsrt decision. self.payment and utility function log MUST BE USED. (later)
    def workchange(self):
        '''ALERT: For now decision is made based on payment + madness \n Changes workplace according to self.madness, *somehow* calculated satisfaction \n Note: will be calld every time period, return True/False if changed/styed'''

        
        #Decision of a work change 
        if self.madness + np.random.normal(loc=0.0, scale=self._brain_damage) > self._brain_damage:
            #Note: I don`t like this condition
            
            ids, payments, _ = self.search_work_offers()
            if len(ids) > 0:
                probs = softmax(payments + payments.mean()*np.random.normal(loc=0.0, scale=abs(self.madness)))
                
                self.model.LabourMarket.get_job(household_id=self.id, company_id=np.random.choice(ids, p=probs))

    
    #future update
    # add log
    def muliply(self):
        # makes a new Household with a small percent according to [ONLY IF IT HAS ENOUGHT SAVINGS]
        # self.madness ang global GOV_SUPPORT
        # returns True if a new Household is made
        # Also 'mothers' capital will be reduced with a global MULTIPLY_COST 
        ...
        

In [32]:
# #pd.Dataframe with precise production functions
# #Данные из:
# #Афанасьев А.А. Использование производственной функции Кобба–Дугласа, построенной по панельным данным, при анализе обрабатывающих производств России // Креативная экономика. – 2022. – Том 16. – № 6. – С. 2363–2380. doi: 10.18334/ce.16.6.114851
# data = [
#     ["Обрабатывающие производства", 2.02, 0.53, 0.51, 0.85],
#     ["Производство пищевых продуктов", 6.14, 0.71, 0.22, 0.69],
#     ["Производство напитков", 4.74, 0.48, 0.47, 0.83],
#     ["Производство табачных изделий", 0.13, 1.67, -0.62, 0.72],
#     ["Производство текстильных изделий", 11.84, 0.46, 0.32, 0.48],
#     ["Производство одежды", 5.77, 0.06, 0.64, 0.16],
#     ["Производство кожи и изделий из кожи", 0.08, -0.05, 1.36, 0.66],
#     ["Обработка древесины и производство изделий из дерева и пробки", 47.69, 0.60, -0.02, 0.54],
#     ["Производство бумаги и бумажных изделий", 3.2, 0.40, 0.62, 0.90],
#     ["Деятельность полиграфическая и копирование носителей информации", 4.75, 0.83, 0.08, 0.95],
#     ["Производство кокса и нефтепродуктов", 15.24, -0.09, 1.25, 0.32],
#     ["Производство химических веществ и химических продуктов", 18.4, 0.59, 0.21, 0.96],
#     ["Производство лекарственных средств и материалов", 10.79, 0.70, 0.14, 0.78],
#     ["Производство резиновых и пластмассовых изделий", 6.12, 0.19, 0.75, 0.76],
#     ["Производство прочей неметаллической минеральной продукции", 1.66, 0.11, 0.94, 0.72],
#     ["Производство металлургическое", 1.38, -0.08, 1.28, 0.79],
#     ["Производство готовых металлических изделий", 1.55, 0.17, 0.88, 0.85],
#     ["Производство компьютеров, электронных и оптических изделий", 7.28, 0.97, -0.06, 0.93],
#     ["Производство электрического оборудования", 13.12, 0.72, 0.13, 0.84],
#     ["Производство машин и оборудования", 0.22, 0.36, 0.90, 0.83],
#     ["Производство автотранспортных средств", 49.2, 0.37, 0.44, 0.35],
#     ["Производство прочих транспортных средств", 28.9, 0.99, -0.28, 0.70],
#     ["Производство мебели", 43896, 0.17, -0.11, -0.16],
#     ["Производство прочих готовых изделий", 7.24, 0.48, 0.47, 0.81],
#     ["Ремонт и монтаж машин и оборудования", 0.4, 0.08, 1.08, 0.73],
# ]

# df = pd.DataFrame(data, columns=["industry", "A", "alpha", "beta", "R2"])

# df[df['R2']>0.84]
# # df.describe().T

In [33]:
import numpy as np
import pandas as pd

EXPORT_PRICE = 100*80
LOGISTICS_EXP = 5.0
GOV_DMO = 0.2
LIFTING_COST_UNIT = 8.5
MAINTENANCE_RATE = 0.05


def q(A, alpha, L, K, elastisity):
    """ CES / Cobb-Douglas """
    if elastisity == 1:
        return A * (K**alpha) * (L**(1 - alpha))

    p = (elastisity - 1) / elastisity
    inner = alpha * (K**p) + (1 - alpha) * (L**p)
    return A * (max(1e-6, inner) ** (1 / p))


class Firm:

    def __init__(self, A, alpha, elastisity, start_capital,
                 price, payment, init_reserves,
                 model, prestigue=1, brain_damage=6.0):

        self.model = model

        # регистрация в системе
        self.id = self.model.resolver.add_firm(self)

        # production
        self.A = A
        self.alpha = alpha
        self.elastisity = elastisity

        # ресурсы
        self.capital = start_capital
        self.init_reserves = init_reserves
        self.reserves = init_reserves

        # поведение
        self.madness = np.random.normal(0.0, brain_damage)
        self.prestigue = prestigue

        # цены и труд
        self.price = price
        self.payment = payment

        # состояние
        self.avaliable_man_hours = 0
        self.unrealised_product = 0
        self.planned_domestic_q = 0
        self.last_step_sold_inner = 0

        # лог данных
        self.InnerPrice  = np.empty(0, dtype=np.float64)
        self.InnerSold   = np.empty(0, dtype=np.float64)
        self.ExportPrice = np.empty(0, dtype=np.float64)
        self.ExportSold  = np.empty(0, dtype=np.float64)
        self.Profit      = np.empty(0, dtype=np.float64)

        # self.time_period = 0
        

    # ---------------- ECONOMICS ---------------- #

    def calculate_taxes(self, price):
        if price > 13.5:
            return 0.644 * (price - 13.5)
        return 0

    def calculate_extraction_costs(self, q_produced):
        depletion = self.init_reserves / max(self.reserves, 1)

        lifting = q_produced * LIFTING_COST_UNIT * depletion
        maintenance = self.capital * MAINTENANCE_RATE

        return lifting + maintenance

    # ---------------- PRODUCTION ---------------- #

    def manufacture(self):

        potential_q = q(
            self.A, self.alpha,
            self.avaliable_man_hours,
            self.capital,
            self.elastisity
        )

        # stochastic shock
        shock = np.clip(
            1 + np.random.normal(0, abs(self.madness) * 0.01),
            0.8, 1.2
        )
        potential_q *= shock

        actual_q = min(
            potential_q,
            self.model.opec_quota / self.model.num_firms,
            self.reserves
        )

        costs = self.calculate_extraction_costs(actual_q)

        self.unrealised_product = actual_q
        self.reserves          -= actual_q
        self.capital           -= costs

        # GlobalResolver.buffer_log()
        self.model.resolver.buffer_log(
            facility_id=2, 
            event_id=2, 
            data_cort=(self.id, actual_q, costs)
        )
        # self.model.resolver.log(
        #     message_cort=(
        #         'Firm',
        #         'Extraction completed',
        #         f'Id: {self.id} Q: {actual_q:.2f} Cost: {costs:.2f}'
        #     )
        # )

    # ---------------- MARKET ---------------- #

    def put_up_for_sale(self):

        self.model.GoodsMarket.push(
            company_id=self.id,
            quantity=self.planned_domestic_q,
            price=self.price,
            prestigue=self.prestigue
        )

        # self.SalesData.loc[len(self.SalesData)] = [
        #     self.price, 0, EXPORT_PRICE, 0, 0
        # ]

        self.InnerPrice  = np.append(self.InnerPrice, self.price)
        self.InnerSold   = np.append(self.InnerPrice, 0)
        self.ExportPrice = np.append(self.InnerPrice, EXPORT_PRICE)
        self.ExportSold  = np.append(self.InnerPrice, 0)
        self.Profit      = np.append(self.InnerPrice, 0)

    def log_solded(self, quantity):
        indx = len(self.InnerSold) - 1 
        self.InnerSold[indx] += quantity

    # ---------------- HR ---------------- #                                                    

    def labor_analytics(self):

        mean_wage = self.model.LabourMarket.get_mean_wage()

        if len(self.Profit) > 0 and self.Profit[-1] > 0:
            self.payment *= (1 + abs(self.madness) * 0.001)
        elif self.capital < 0:
            self.payment *= 0.98

        vacancies = max(1, int(5 + self.madness))                 # total fuck, ни единой аналитики

        self.model.LabourMarket.push(
            company_id= self.id,
            vacancies=vacancies,
            payment=self.payment
        )

    # ---------------- STEPS ---------------- #

    def step_before_Households(self):

        self.labor_analytics()
        self.manufacture()

        predicted = self.last_step_sold_inner * 1.05
        legal_min = self.unrealised_product * GOV_DMO

        self.planned_domestic_q = max(predicted, legal_min)

        self.put_up_for_sale()
        

    def step_after_Households(self):

        indx = len(self.InnerSold)-1
    
        self.last_step_sold_inner = self.InnerSold[-1]

        inner_rev = self.last_step_sold_inner * self.InnerPrice[-1]

        export_q = self.unrealised_product

        tax = self.calculate_taxes(EXPORT_PRICE)
        netback = EXPORT_PRICE - tax - LOGISTICS_EXP

        export_rev = export_q * netback * self.model.usd_rate

        damper = max(
            0,
            (netback - (self.price / self.model.usd_rate)) * 0.5
        )
        damper_total = self.last_step_sold_inner * damper * self.model.usd_rate

        fot = self.avaliable_man_hours * self.payment

        profit = inner_rev + export_rev + damper_total - fot
        self.capital += profit

        self.ExportSold[-1] = export_q
        self.Profit[-1] = profit

        # GlobalResolver.buffer_log()
        self.model.resolver.buffer_log(
            facility_id=2, 
            event_id=4, 
            data_cort=(self.id, profit)
        )
        # self.model.resolver.log(
        #     message_cort=(
        #         'Firm',
        #         'Sales completed',
        #         f'Id: {self.id} Profit: {profit:.2f}'
        #     )
        # )

    # ---------------- INFO ---------------- #

    def get_last_step_sales(self):
        return self.last_step_sold_inner

    def info(self):
        return pd.DataFrame([{
            "Capital": self.capital,
            "Company ID": self.id,
            "Madness": self.madness,
            "Prestige": self.prestigue,
            "Price": self.price,
            "Payment": self.payment,
            "Available Man Hours": self.avaliable_man_hours,
            "Unrealised Product": self.unrealised_product,
            "Planned Domestic Q": self.planned_domestic_q,
            "Reserves": self.reserves
        }])

In [34]:
import os

class model:
    
    def __init__(self, num_firms, num_households, logging_goods = True, no_complete_sale=False):
        self.num_firms       = num_firms
        self.num_households  = num_households

        self.resolver        = GlobalResolver(self)
        self.LabourMarket    = LabourMarket(self.resolver)
        self.GoodsMarket     = GoodsMarket(self.resolver, logging_goods, no_compete_sale=no_complete_sale)

        for i in range(num_firms):
            Firm(
                A = 1.8,              # Технологический коэффициент (TFP)
                alpha = 0.65,         # Доля капитала в добыче (отрасль крайне капиталоемкая)
                elastisity = 0.5,     # Низкая эластичность замены труда капиталом на скважинах
                start_capital = 1e11, # ~100 млрд руб (начальный баланс крупной компании)
                price = 5000,         # Внутренняя цена реализации (руб)
                payment = 102000,     # Средняя зп по отрасли (руб/мес)
                init_reserves = 5e8,  # 500 млн баррелей (ресурсная база агента)
                model = self,
                prestigue = 1.0,
                brain_damage = 4.0    # Умеренная волатильность решений руководства
            )

        for _ in range(num_households):
            Household(
                # Начальные сбережения: ~4 месяца зарплаты
                start_savings = 400000, 
                
                # Случайный выбор работодателя среди созданных фирм
                company_id = np.random.choice(
                    a = list(self.resolver.companies.keys()),
                    size = 1,
                    p = [1/num_firms] * num_firms
                ),
                
                # Базовая зп соответствует отраслевой для сотрудников ВИНК
                payment = 102000, 
                model = self,
                brain_damage = 6.0 # Индекс Madness (склонность к шуму в потреблении)
            )                         

    def set_job_params(self, BASE_JOB_OFFERS_AMOUNT, START_MEAN_WAGE):
        self.BASE_JOB_OFFERS_AMOUNT = BASE_JOB_OFFERS_AMOUNT
        self.START_MEAN_WAGE        = START_MEAN_WAGE

    def set_export_params(self, opec_quota, p_brent, discount_urals, usd_rate, logistics_exp):
        self.opec_quota     = opec_quota
        self.p_brent        = p_brent
        self.discount_urals = discount_urals
        self.usd_rate        = usd_rate
        self.logistics_exp  = logistics_exp

    def set_gov_params(self, gas_price_limit ):
        self.gas_price_limit = gas_price_limit


    def step(self):
        self.resolver.calculate_manhours()
        
        self.resolver.step_before_Households()        
        self.resolver.step_consume_Households()
        self.resolver.step_after_Households()
        
        self.resolver.step_workchange_Households()
        self.resolver.flush_to_csv('/home/makra/a_unik/ABM/DataLog/testlog.csv')


In [39]:
model_s   = model(num_firms = 5, num_households=int(1000000), logging_goods=True, no_complete_sale=False)
model_ncs = model(num_firms = 5, num_households=int(10000), logging_goods=True, no_complete_sale=False)

# Вызов функций установки параметров рынка
# opec_quota: ~9.5 млн барр./сутки (общая квота РФ в 2025)
# p_brent: $74.2 (средний прогноз EIA/WoodMac на 2025)
# discount_urals: $25.0 (средний санкционный дисконт)
# usd_rub: 92.0 (прогнозный курс для расчетов бюджета)
# logistics_exp: $18.0 (высокие затраты на обход санкций)

model_s.set_export_params(
    opec_quota = 9500000, 
    p_brent = 74.2*80, 
    discount_urals = 25.0*80, 
    usd_rate = 92.0*80, 
    logistics_exp = 18.0*80
)

model_ncs.set_export_params(
    opec_quota = 9500000, 
    p_brent = 74.2*80, 
    discount_urals = 25.0*80, 
    usd_rate = 92.0*80, 
    logistics_exp = 18.0*80
)

# gas_price_limit: 65.0 руб (розничная цена бензина/газа с учетом индексации 10.3%)
model_s.set_gov_params(gas_price_limit = 5000)
model_s.set_job_params(
    BASE_JOB_OFFERS_AMOUNT=3,
    START_MEAN_WAGE = 102000
)

model_ncs.set_gov_params(gas_price_limit = 5000)
model_ncs.set_job_params(
    BASE_JOB_OFFERS_AMOUNT=3,
    START_MEAN_WAGE = 102000
)

/tmp/ipykernel_6560/2294615152.py:83: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  self.workplace_id  = int(company_id)
/tmp/ipykernel_6560/2294615152.py:83: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  self.workplace_id  = int(company_id)


In [40]:
# model_s.resolver.companies_info()

In [41]:

# model_ncs.resolver.companies_info()


In [42]:
import time

for i in range(1):
    start1 = time.time()
    
    model_s.step()
    # print(i, model_s.resolver.households_info()['Savings'].sum())
    end1 = time.time()

    print(f"Execution time: {end1 - start1:.6f} seconds")
    

# start2 = time.time()

# model_ncs.step()

# end2 = time.time()

Execution time: 50.917616 seconds


In [43]:
data = pd.read_csv('/home/makra/a_unik/ABM/DataLog/testlog.csv')
data

,Facility,Message Type,Message_txt
0,Labour Market,Job offer listed,Id: 1 vacancies=5 payment=102000
1,Firm,Extraction completed,Id: 1 Q: 1628027.62 Cost: 5013838234.75
2,Goods Market,Goods listed,Id: 1 listed 325605.52347497793 Price: 5000
3,Labour Market,Job offer listed,Id: 2 vacancies=3 payment=102000
4,Firm,Extraction completed,Id: 2 Q: 1664304.51 Cost: 5014146588.30
...,...,...,...
950134,Household,Worplace change: success,Id: 111 changed workplace to 4
950135,Household,Worplace change: success,Id: 112 changed workplace to 4
950136,Household,Worplace change: success,Id: 114 changed workplace to 4
950137,Household,Worplace change: success,Id: 116 changed workplace to 4


In [ ]:
print(f"Execution time: {end1 - start1:.6f} seconds")
model_s.resolver.households_info()['Savings'].sum()

# print(f"Execution time: {end2 - start2:.6f} seconds")

In [ ]:
model_s.LabourMarket.actual_data()

In [ ]:
model_s.LabourMarket.payments

In [ ]:
model_s.LabourMarket.id_to_idx

In [ ]:
model_s.resolver.show_log()['Message Type'].unique()

In [ ]:
model_s.resolver.households_info()

In [ ]:
model_s.resolver.companies_info()

In [ ]:
model_ncs.resolver.companies_info()


In [ ]:
model_s.resolver.show_log()[model_s.resolver.show_log()['Facility']=='Goods Market'].loc[15][2]

In [ ]:
assert model_s.resolver.companies_info()['Workers'].sum() == model_s.num_households

In [ ]:
model_s.resolver.households_info()

In [ ]:

df = model_s.resolver.show_log()
spent     = df[df['Message Type']=='Consumed']['Message_txt'].apply(lambda x: float(x.split()[-1])).sum()
start_cap = model_s.num_households * 400000
savings   = model_s.resolver.households_info()['Savings'].sum()

assert (savings + spent) == start_cap

In [ ]:

df = model_ncs.resolver.show_log()
spent     = df[df['Message Type']=='Consumed']['Message_txt'].apply(lambda x: float(x.split()[-1])).sum()
start_cap = model_ncs.num_households * 400000
savings   = model_ncs.resolver.households_info()['Savings'].sum()

assert (savings + spent) == start_cap

In [ ]:
int(savings + spent), start_cap, int(savings + spent)- start_cap

In [ ]:
model_ncs.resolver.households_info()

In [ ]:
model_s.resolver.households[80].deals

In [ ]:
[model_s.resolver.companies[i].payment for i in range(1, model_s.num_firms+1)]

In [ ]:
pd.concat([model_s.resolver.companies[i].SalesData for i in range(1, model_s.num_firms+1)])

In [ ]:
model_s.resolver.companies_info()

In [ ]:
df = model_s.resolver.show_log()
df[df['Facility'] == 'Household']

In [ ]:
model_s.GoodsMarket.actual_data()

In [ ]:
model_s.LabourMarket.actual_data()

In [ ]:
model_s.resolver.households_info()

In [ ]:
model_s.resolver.households_info()

In [ ]:
model_s.resolver.companies_info()

In [ ]:
# model_s.resolver.calculate_manhours()

In [ ]:
nn = np.arange(10)
nn


In [ ]:
nn[-1]